In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

In [3]:
df = pd.read_csv(r'pandas/data/computer_prices_all.csv')
df.head()

,device_type,brand,model,release_year,os,form_factor,cpu_brand,cpu_model,cpu_tier,cpu_cores,...,resolution,refresh_hz,battery_wh,charger_watts,psu_watts,wifi,bluetooth,weight_kg,warranty_months,price
0,Desktop,Samsung,Samsung Forge XDI,2022,Windows,ATX,Intel,Intel i5-11129,3,12,...,2560x1440,90,0,0,750,Wi-Fi 6,5.1,11.00,36,1383.99
1,Laptop,Samsung,Samsung Pro KM8,2022,Windows,Mainstream,Intel,Intel i7-11114,4,12,...,1920x1080,90,56,120,0,Wi-Fi 6,5.3,2.03,12,2274.99
2,Desktop,Lenovo,Lenovo Strix BIE,2024,macOS,SFF,AMD,AMD Ryzen 5 5168,2,8,...,3440x1440,120,0,0,850,Wi-Fi 6,5.0,7.00,24,1879.99
3,Desktop,Dell,Dell Cube AXR,2024,Windows,ATX,AMD,AMD Ryzen 5 7550,2,6,...,3440x1440,120,0,0,650,Wi-Fi 6,5.2,6.00,36,1331.99
4,Laptop,Gigabyte,Gigabyte Pro IX1,2024,Linux,Gaming,AMD,AMD Ryzen 7 6230,5,16,...,2560x1600,90,80,90,0,Wi-Fi 6,5.2,1.50,12,2681.99


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 33 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   device_type          100000 non-null  object 
 1   brand                100000 non-null  object 
 2   model                100000 non-null  object 
 3   release_year         100000 non-null  int64  
 4   os                   100000 non-null  object 
 5   form_factor          100000 non-null  object 
 6   cpu_brand            100000 non-null  object 
 7   cpu_model            100000 non-null  object 
 8   cpu_tier             100000 non-null  int64  
 9   cpu_cores            100000 non-null  int64  
 10  cpu_threads          100000 non-null  int64  
 11  cpu_base_ghz         100000 non-null  float64
 12  cpu_boost_ghz        100000 non-null  float64
 13  gpu_brand            100000 non-null  object 
 14  gpu_model            100000 non-null  object 
 15  gpu_tier          

In [4]:
# 결측치 확인
df.isnull().sum()

device_type            0
brand                  0
model                  0
release_year           0
os                     0
form_factor            0
cpu_brand              0
cpu_model              0
cpu_tier               0
cpu_cores              0
cpu_threads            0
cpu_base_ghz           0
cpu_boost_ghz          0
gpu_brand              0
gpu_model              0
gpu_tier               0
vram_gb                0
ram_gb                 0
storage_type           0
storage_gb             0
storage_drive_count    0
display_type           0
display_size_in        0
resolution             0
refresh_hz             0
battery_wh             0
charger_watts          0
psu_watts              0
wifi                   0
bluetooth              0
weight_kg              0
warranty_months        0
price                  0
dtype: int64

In [6]:
# model, cpu_model은 제거
df_copy = df.drop(['model', 'cpu_model'], axis=1)
df_copy.head()

,device_type,brand,release_year,os,form_factor,cpu_brand,cpu_tier,cpu_cores,cpu_threads,cpu_base_ghz,...,resolution,refresh_hz,battery_wh,charger_watts,psu_watts,wifi,bluetooth,weight_kg,warranty_months,price
0,Desktop,Samsung,2022,Windows,ATX,Intel,3,12,24,2.8,...,2560x1440,90,0,0,750,Wi-Fi 6,5.1,11.00,36,1383.99
1,Laptop,Samsung,2022,Windows,Mainstream,Intel,4,12,24,2.6,...,1920x1080,90,56,120,0,Wi-Fi 6,5.3,2.03,12,2274.99
2,Desktop,Lenovo,2024,macOS,SFF,AMD,2,8,16,2.6,...,3440x1440,120,0,0,850,Wi-Fi 6,5.0,7.00,24,1879.99
3,Desktop,Dell,2024,Windows,ATX,AMD,2,6,12,2.6,...,3440x1440,120,0,0,650,Wi-Fi 6,5.2,6.00,36,1331.99
4,Laptop,Gigabyte,2024,Linux,Gaming,AMD,5,16,32,2.8,...,2560x1600,90,80,90,0,Wi-Fi 6,5.2,1.50,12,2681.99


In [7]:
df_num_cols = df_copy.select_dtypes('number')
df_num_cols.head()

,release_year,cpu_tier,cpu_cores,cpu_threads,cpu_base_ghz,cpu_boost_ghz,gpu_tier,vram_gb,ram_gb,storage_gb,storage_drive_count,display_size_in,refresh_hz,battery_wh,charger_watts,psu_watts,bluetooth,weight_kg,warranty_months,price
0,2022,3,12,24,2.8,3.8,2,6,16,1024,1,27.0,90,0,0,750,5.1,11.00,36,1383.99
1,2022,4,12,24,2.6,3.6,4,10,64,512,1,16.0,90,56,120,0,5.3,2.03,12,2274.99
2,2024,2,8,16,2.6,3.6,1,4,8,512,2,32.0,120,0,0,850,5.0,7.00,24,1879.99
3,2024,2,6,12,2.6,3.6,2,6,16,512,2,27.0,120,0,0,650,5.2,6.00,36,1331.99
4,2024,5,16,32,2.8,3.9,5,12,96,256,1,15.6,90,80,90,0,5.2,1.50,12,2681.99


In [15]:
def print_outlier(data:pd.Series):
    iqr1 = data.quantile(0.25)
    iqr3 = data.quantile(0.75)
    iqr = iqr3 - iqr1
    upper_bound = iqr3 + iqr * 1.5
    lower_bound = iqr1 - iqr * 1.5
    
    data_upper = data[data.values > upper_bound]
    data_lower = data[data.values < lower_bound]
    
    upper_out_count = data_upper.count()
    lower_out_count = data_lower.count()
    
    print(data.name, f'행개수: {len(data)}개')
    print(f'IQR1: {iqr1}')
    print(f'IQR3: {iqr3}')
    print(f'IQR: {iqr}')
    
    print(f'경계 최대값: {upper_bound}')
    print(f'경계 최소값: {lower_bound}')
    
    print(f'상한 이상치 : {upper_out_count} 개')
    print(f'하한 이상치 : {lower_out_count} 개')
    
    print(f'상한 이상치 데이터(index): {data_upper.index}')
    print(f'하한 이상치 데이터(index): {data_lower.index}')
    
    
    
print_outlier(df_num_cols['price'])

price 행개수: 100000개
IQR1: 1503.99
IQR3: 2287.99
IQR: 783.9999999999998
경계 최대값: 3463.9899999999993
경계 최소값: 327.99000000000046
상한 이상치 : 976 개
하한 이상치 : 0 개
상한 이상치 데이터(index): Index([   26,   149,   203,   208,   219,   632,   664,   671,   777,  1021,
       ...
       99168, 99208, 99234, 99298, 99358, 99367, 99665, 99774, 99868, 99967],
      dtype='int64', length=976)
하한 이상치 데이터(index): Index([], dtype='int64')


In [16]:
df_obj_cols = df_copy.drop(df_num_cols.columns, axis=1)
print(df_num_cols.columns.to_list())
print(df_obj_cols.columns.to_list())

['release_year', 'cpu_tier', 'cpu_cores', 'cpu_threads', 'cpu_base_ghz', 'cpu_boost_ghz', 'gpu_tier', 'vram_gb', 'ram_gb', 'storage_gb', 'storage_drive_count', 'display_size_in', 'refresh_hz', 'battery_wh', 'charger_watts', 'psu_watts', 'bluetooth', 'weight_kg', 'warranty_months', 'price']
['device_type', 'brand', 'os', 'form_factor', 'cpu_brand', 'gpu_brand', 'gpu_model', 'storage_type', 'display_type', 'resolution', 'wifi']
